# ASTRID Vignette — runnable local copy

Same cells, same order, same flow as `ASTRID_Vignette.ipynb`. Only three things changed,
each marked `# CHANGED`:

1. **Paths** → your machine (`adrian_spec/forAdrian 2/`, outputs to `adrian_spec/vignette_run/`)
2. **Cell 7's `os.system(f'...')`** → the original is a multi-line f-string, which is a
   `SyntaxError`; replaced with a `subprocess` call that also streams progress into the cell
3. **Patient** is a variable — the original comment says p2 but the code selects `p1`

Defaults reproduce the run whose outputs are **stored in the committed vignette** (p1, immune
subset), so you can check your numbers against Alper's. The last cell does that automatically.

**Runtime:** ~1.5 min to load the raw matrix, ~9 min for ASTRID. **Kernel:** ASTRID (.venv310)


In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.rcParams['figure.figsize']=(5,5) #rescale figures
plt.rcParams['pdf.fonttype']=42 #for vectorized text in pdfs
sns.set_theme(style="white")

# CHANGED: local settings. cwd must be the repo root because RunSingleR.R loads its
# reference with a relative path ("data/ASTRID_SingleR_Reference_20240701.Rds").
import sys, subprocess, time, warnings
from pathlib import Path

REPO = Path("/Users/adriansohrabi/Documents/GitHub/astrid")
os.chdir(REPO)

DATA    = REPO / "adrian_spec/forAdrian 2"        # the three GSE127465 files
OUT     = REPO / "adrian_spec/vignette_run"       # everything this notebook writes
PATIENT = "p1"                                    # p1 matches the stored reference outputs
USE_CACHE = False                                 # True reuses a cached copy of cell 2

OUT.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=FutureWarning)
assert (REPO / "data/ASTRID_SingleR_Reference_20240701.Rds").exists(), "wrong cwd"
print("cwd:", os.getcwd(), "| patient:", PATIENT)

cwd: /Users/adriansohrabi/Documents/GitHub/astrid | patient: p1


/Users/adriansohrabi/Documents/GitHub/astrid/.venv310/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/Users/adriansohrabi/Documents/GitHub/astrid/.venv310/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/Users/adriansohrabi/Documents/GitHub/astrid/.venv310/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/Users/adriansohrabi/Documents/GitHub/astrid/.venv310/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.
  warnings.warn(msg, FutureWarning)
/Users/adriansohrabi/Documents

In [ ]:
# load in data from Zillionis et al 2019
# CHANGED: paths point at adrian_spec/forAdrian 2/ instead of the hausserlab cluster.
# The mtx has 44.7M non-zeros; the read takes ~80 s and peaks around 2.7 GB of RAM.
# A cached copy already exists in vignette_run/ if you ever want to skip it (USE_CACHE=True);
# it is the same object round-tripped through h5ad, so string columns come back categorical.
cache = OUT / "_cache_immune.h5ad"

if USE_CACHE and cache.exists():
    adata = sc.read_h5ad(cache)
    print("loaded from cache:", cache.name)
else:
    t0 = time.time()
    adata = sc.read_mtx(DATA / 'GSE127465_human_counts_normalized_54773x41861.mtx.gz')

    # add gene names
    adata.var_names = np.loadtxt(DATA / 'GSE127465_gene_names_human_41861.tsv.gz', dtype=str)
    # add per-cell metadata
    adata.obs = pd.read_csv(DATA / 'GSE127465_human_cell_metadata_54773x25.tsv.gz', sep='\t')

    # this data contain already log1p normalized counts, we want the raw versions, so we will
    # transform the data by reversing the normalization based on author provided total UMIs
    sc.pp.normalize_total(adata, target_sum=adata.obs['Total counts'].values)

    # as this is a too large with many tissues, we will only focus on one of the tissues
    # provided by authors
    adata = adata[adata.obs['used_in_NSCLC_immune'].astype(bool).values]
    adata.var['mt'] = adata.var_names.str.startswith('MT-')   # mitochondrial genes
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

    adata.write_h5ad(cache)
    print(f"read from raw in {(time.time()-t0)/60:.1f} min, cached to {cache.name}")

adata

In [ ]:
adata.obs

In [ ]:
# lets pick a patient from this to make our life easier
# CHANGED: the original comment says p2 but the code selects p1. PATIENT is set in cell 1.
adata = adata[adata.obs["Patient"] == PATIENT]

adata

In [ ]:
adata.obs["log10_UMIs"] = np.log10(adata.obs["total_counts"])
plt.rcParams['figure.figsize']=(10,5) #rescale figures

plt.subplot(1, 2, 1)
sns.histplot(adata.obs["log10_UMIs"], bins=100)

plt.subplot(1, 2, 2)
adata.obs["pct_counts_mt"].plot.hist(bins=100)
plt.axvline(x=10, color='r', linestyle='--')

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

# filter out cells with more than 15% of their transciptome is from mitochondria genes
adata = adata[adata.obs["pct_counts_mt"]<15]

adata

In [ ]:
# store the user provided cell types in a new column of .obs called cellType
adata.obs["cellType"] = adata.obs["Major cell type"]

# let's check how many cells we have for each subset
adata.obs["cellType"].value_counts()

In [ ]:
# to define the parameters of ASTRID easier, we'll provide the sample name
sample = f"Zilionis_{PATIENT}_ASTRID_Example"

# CHANGED: all four paths now point into adrian_spec/vignette_run/
adata_file        = OUT / f"{sample}.h5ad"                 # pre-ASTRID input
prefix            = sample
output_file       = OUT / f"{sample}_outASTRID.h5ad"       # output AnnData
clustering_result = OUT / f"{sample}_ASTRID_Result.csv"    # clustering + annotation result
author_type       = 'cellType'                             # author labels, from adata.obs
output_directory  = OUT                                    # plots (needs --plot to fill)

# we will first write this Anndata object
adata.write_h5ad(adata_file)
print("wrote", adata_file.name, adata.shape)

# IMPORTANT: for mouse data add this option --skip_cell_typing as there is no reference for mice
# --out_dir gives the directory for the clustering and validation plots
#
# CHANGED: the original os.system(f'...') is a single-quoted f-string broken across 8 lines,
# which is a SyntaxError. subprocess also puts the output in THIS cell — os.system writes to
# the terminal that launched the kernel, where you would never see it.
cmd = [
    sys.executable, "ASTRID_v0.01.py",
    "--clustering", "--annotation", "--validation",
    "--input_file",                str(adata_file),
    "--input_prefix",              prefix,
    "--output_file",               str(output_file),
    "--output_clustering_results", str(clustering_result),
    "--author_type",               author_type,
    "--out_dir",                   str(output_directory),
]
t0 = time.time()
proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
with open(OUT / "run.log", "w") as log:
    for line in proc.stdout:
        log.write(line)
        if "FutureWarning" in line or "warnings.warn" in line:
            continue
        print(line, end="")
rc = proc.wait()
print(f"\nexit {rc} in {(time.time()-t0)/60:.1f} min")

# in the end we will read in the data from the output
tmp_data = sc.read_h5ad(output_file)

tmp_data

In [ ]:
plt.rcParams['figure.figsize']=(5,5) #rescale figures
sc.pl.umap(tmp_data, color = ["Major cell type", "SingleR_CellType"], legend_loc="on data")

In [ ]:
# let's read the output file from ASTRID
astridSummary = pd.read_csv(clustering_result)

astridSummary

In [ ]:
# CHANGED (new cell): check your run against the outputs stored in the committed vignette.
# Those came from Alper's machine on p1 with the same settings, so the numbers should match
# closely. Small drifts are normal; a big gap means something differs in the environment.
ref = {"cells": 4393, "clusters": 208, "nmi_level1": 0.7533, "nmi_final": 0.9551,
       "nmi_singler": 0.6800, "passed": 101}

log = (OUT / "run.log").read_text()
import re
def grab(pat, default=float("nan")):
    m = re.search(pat, log)
    return float(m.group(1)) if m else default

mine = {
    "cells":       tmp_data.n_obs,
    "clusters":    int(grab(r"Number of clusters: (\d+)\s*$", 0) or astridSummary.shape[0]),
    "nmi_level1":  grab(r"clustering_level_1 and cellType: ([\d.]+)"),
    "nmi_final":   grab(r"clustering_level_11 and cellType: ([\d.]+)"),
    "nmi_singler": grab(r"SingleR_CellType and cellType: ([\d.]+)"),
    "passed":      int(grab(r"passed the formula out of the applicable/total: (\d+)", 0)),
}
mine["clusters"] = len(astridSummary)

cmp = pd.DataFrame({"reference (Alper)": ref, "this run": mine})
cmp["delta"] = (pd.to_numeric(cmp["this run"]) - pd.to_numeric(cmp["reference (Alper)"])).round(4)
cmp

### What you just did

1. Rebuilt the input from the raw GSE127465 matrix (54,773 cells) → immune subset (34,558)
   → one patient → QC filters.
2. Ran three of ASTRID's four stages. The vignette omits `--damage`, so the result table has
   no `newCNVScore*` / `CNVCorrelation` columns — add `"--damage"` to `cmd` if you want them.
3. Compared against the reference numbers stored in the committed notebook.

To run p2 through this same flow, set `PATIENT = "p2"` in cell 1 and re-run from cell 3.
Be aware that the `used_in_NSCLC_immune` filter in cell 2 keeps only **345 of p2's 5,166
cells** — that is the tumor-infiltrating immune compartment, not the blood. It is a valid
run, just a very different sample from the 5,166-cell p2 run in `adrian_spec/test_run/`.
